Machine Learning based credit card approval system

Data & Metadata

Target Variable:One

Features:Fifteen

In [1]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo

credit_approval = fetch_ucirepo(id=27)

X = credit_approval.data.features
y = credit_approval.data.targets

print(credit_approval.metadata)

print(credit_approval.variables)



{'uci_id': 27, 'name': 'Credit Approval', 'repository_url': 'https://archive.ics.uci.edu/dataset/27/credit+approval', 'data_url': 'https://archive.ics.uci.edu/static/public/27/data.csv', 'abstract': 'This data concerns credit card applications; good mix of attributes', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 690, 'num_features': 15, 'feature_types': ['Categorical', 'Integer', 'Real'], 'demographics': [], 'target_col': ['A16'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1987, 'last_updated': 'Wed Aug 23 2023', 'dataset_doi': '10.24432/C5FS30', 'creators': ['J. R. Quinlan'], 'intro_paper': None, 'additional_info': {'summary': 'This file concerns credit card applications.  All attribute names and values have been changed to meaningless symbols to protect confidentiality of the data.\r\n  \r\nThis dataset is interesting because there is a good mix of attributes --

Data Preprocessing




In [2]:
print(X.isnull().sum())

A15     0
A14    13
A13     0
A12     0
A11     0
A10     0
A9      0
A8      0
A7      9
A6      9
A5      6
A4      6
A3      0
A2     12
A1     12
dtype: int64


In [3]:
import numpy as np

X = X.replace("?", np.nan)

numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object', 'category']).columns

print("Numerical:", numerical_cols)
print("Categorical:", categorical_cols)


from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='median')
X[numerical_cols] = num_imputer.fit_transform(X[numerical_cols])

cat_imputer = SimpleImputer(strategy='most_frequent')
X[categorical_cols] = cat_imputer.fit_transform(X[categorical_cols])


print(X.isnull().sum())

Numerical: Index(['A15', 'A14', 'A11', 'A8', 'A3', 'A2'], dtype='object')
Categorical: Index(['A13', 'A12', 'A10', 'A9', 'A7', 'A6', 'A5', 'A4', 'A1'], dtype='object')
A15    0
A14    0
A13    0
A12    0
A11    0
A10    0
A9     0
A8     0
A7     0
A6     0
A5     0
A4     0
A3     0
A2     0
A1     0
dtype: int64


Encode variables

In [4]:
from sklearn.preprocessing import LabelEncoder

for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

if hasattr(y, "squeeze"):
    y = y.squeeze()

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)

Train/Test Data

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Logistic Regression

In [8]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(
    max_iter=1000,
    random_state=42
)

log_reg.fit(X_train, y_train)


y_pred = log_reg.predict(X_test)


y_prob = log_reg.predict_proba(X_test)[:, 1]

In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    classification_report
)

y_pred = log_reg.predict(X_test)
y_prob = log_reg.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
mcc = matthews_corrcoef(y_test, y_pred)

print("Logistic Regression Results")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("ROC-AUC  :", round(roc_auc, 4))
print("MCC Score:", round(mcc, 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Logistic Regression Results
Accuracy : 0.8043
Precision: 0.8676
Recall   : 0.7662
F1 Score : 0.8138
ROC-AUC  : 0.8816
MCC Score: 0.6146

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.85      0.79        61
           1       0.87      0.77      0.81        77

    accuracy                           0.80       138
   macro avg       0.81      0.81      0.80       138
weighted avg       0.81      0.80      0.80       138



Desicion Tree

In [23]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    criterion='gini',
    random_state=42,
    max_depth=None
)

dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)

y_prob = dt_model.predict_proba(X_test)[:, 1]

In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
mcc = matthews_corrcoef(y_test, y_pred)

print("Decision Tree Results")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"MCC Score: {mcc:.4f}")
print("\nClassification Report")
print(classification_report(y_test, y_pred))

Decision Tree Results
Accuracy : 0.7899
Precision: 0.8529
Recall   : 0.7532
F1 Score : 0.8000
ROC-AUC  : 0.7947
MCC Score: 0.5854

Classification Report
              precision    recall  f1-score   support

           0       0.73      0.84      0.78        61
           1       0.85      0.75      0.80        77

    accuracy                           0.79       138
   macro avg       0.79      0.79      0.79       138
weighted avg       0.80      0.79      0.79       138



K Nearest Neighbour Classifer

In [25]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(
    n_neighbors=5,
    metric='minkowski',
    p=2
)

knn_model.fit(X_train, y_train)

y_pred = knn_model.predict(X_test)

y_prob = knn_model.predict_proba(X_test)[:, 1]

In [18]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
mcc = matthews_corrcoef(y_test, y_pred)

print("K-Nearest Neighbors Results")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"MCC Score: {mcc:.4f}")
print("\nClassification Report")
print(classification_report(y_test, y_pred))

K-Nearest Neighbors Results
Accuracy : 0.7971
Precision: 0.8101
Recall   : 0.8312
F1 Score : 0.8205
ROC-AUC  : 0.8417
MCC Score: 0.5875

Classification Report
              precision    recall  f1-score   support

           0       0.78      0.75      0.77        61
           1       0.81      0.83      0.82        77

    accuracy                           0.80       138
   macro avg       0.79      0.79      0.79       138
weighted avg       0.80      0.80      0.80       138



Gaussian Naive Bayes Classifier

In [26]:
from sklearn.naive_bayes import GaussianNB

gnb_model = GaussianNB()

gnb_model.fit(X_train, y_train)

y_pred = gnb_model.predict(X_test)

y_prob = gnb_model.predict_proba(X_test)[:, 1]



In [27]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
mcc = matthews_corrcoef(y_test, y_pred)

print("Gaussian Naive Bayes Results")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"MCC Score: {mcc:.4f}")
print("\nClassification Report")
print(classification_report(y_test, y_pred))

Gaussian Naive Bayes Results
Accuracy : 0.8261
Precision: 0.7912
Recall   : 0.9351
F1 Score : 0.8571
ROC-AUC  : 0.8610
MCC Score: 0.6535

Classification Report
              precision    recall  f1-score   support

           0       0.89      0.69      0.78        61
           1       0.79      0.94      0.86        77

    accuracy                           0.83       138
   macro avg       0.84      0.81      0.82       138
weighted avg       0.84      0.83      0.82       138



Random Forest

In [32]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

y_prob = rf_model.predict_proba(X_test)[:, 1]

import joblib
joblib.dump(rf_model, "credit_approval_model.pkl")

['credit_approval_model.pkl']

In [31]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
mcc = matthews_corrcoef(y_test, y_pred)

print("Random Forest Results")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"MCC Score: {mcc:.4f}")
print("\nClassification Report")
print(classification_report(y_test, y_pred))

Random Forest Results
Accuracy : 0.8333
Precision: 0.8553
Recall   : 0.8442
F1 Score : 0.8497
ROC-AUC  : 0.9142
MCC Score: 0.6628

Classification Report
              precision    recall  f1-score   support

           0       0.81      0.82      0.81        61
           1       0.86      0.84      0.85        77

    accuracy                           0.83       138
   macro avg       0.83      0.83      0.83       138
weighted avg       0.83      0.83      0.83       138



In [33]:
import os

print(os.path.exists("/content/credit_approval_model.pkl"))

True


In [34]:
from google.colab import files

files.download("/content/credit_approval_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
import joblib

model = joblib.load("/content/credit_approval_model.pkl")

print(model)

RandomForestClassifier(n_jobs=-1, random_state=42)
